# SAGE (CVPR 2025) — Baseline pretrained Student trên FMB | Quân

**Mục tiêu:** kế thừa notebook smoke 10 ảnh của Lộc bé, chạy **toàn bộ 280 cặp FMB test** trên Kaggle, lưu ảnh fused, manifest, cấu hình, log inference và kiểm nghiệm output.

**Đường dẫn FMB test Quân đã xác nhận:** `/kaggle/input/datasets/quannh206/data-fmbtest/test`.

> **QUAN TRỌNG VỀ SPLIT:** `train` thường có **1.220 cặp**; `test` thường có **280 cặp**. Chạy trên `train` **không phải** tái lập benchmark test chính thức. Notebook **cố định** ở `test` (280 cặp) để chạy benchmark FMB test; không tự chuyển sang train khi không tìm thấy đường dẫn. Nếu dữ liệu không đủ 280 ảnh, notebook dừng để bạn kiểm tra lại bản dataset. Không dùng kết quả test để chọn solution/hyperparameters.

**Nguồn:** [SAGE official](https://github.com/RollingPlain/SAGE_IVIF) · [Paper CVPR 2025](https://openaccess.thecvf.com/content/CVPR2025/papers/Wu_Every_SAM_Drop_Counts_Embracing_Semantic_Priors_for_Multi-Modality_Image_CVPR_2025_paper.pdf)

**Kaggle:** `Settings → Accelerator → GPU`; `Internet → On` **nếu cần clone repo**. Dataset phải được Add Input. Bấm **Run All** sau khi kiểm tra cấu hình ở ô kế tiếp.

**Phạm vi đạt được:** pretrained Student inference + kiểm tra đầu ra + thời gian. **Chưa** phải tái lập toàn bộ Teacher training hoặc mIoU 61.2% (cần SegFormer-B3 evaluator riêng). Các chỉ số EN/SD/SCD/MS-SSIM do pipeline evaluation của Lộc lớn tính sau.

## 1. Cấu hình — FMB test chính thức (280 cặp)

Notebook này chỉ chạy split `test` từ đường dẫn bạn cung cấp. `MAX_IMAGES=None` nghĩa là chạy đủ 280 cặp; không chuyển sang train để thay thế nếu thiếu dữ liệu.

**Mục đích:** tái lập output inference của **pretrained SAGE Student** trên toàn FMB test, lưu mốc đối chứng (checkpoint, preprocessing, fused PNG, runtime, QA). Đây chưa phải tái lập `61.2% mIoU` vì còn cần evaluator segmentation. **Không xem/khai thác test để chọn solution hoặc điều chỉnh hyperparameter**; nghiên cứu và tuning phải dùng dev tách từ train.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, random, hashlib, shutil, subprocess, time, platform
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ===== ĐƯỜNG DẪN TEST DO QUÂN CUNG CẤP =====
FMB_TEST_ROOT = Path('/kaggle/input/datasets/quannh206/data-fmbtest/test')
RUN_SPLIT = 'test'        # FMB test = 280 cặp, không đổi thành train
MAX_IMAGES = None        # None = chạy tất cả; 10 = smoke (KHÔNG gọi là full baseline)
STRICT_COUNT = True      # Bật nếu bạn sử dụng FMB 1220/280 chính thức
EXPORT_ZIP = True       # Tạo ZIP chứa ảnh PNG và log, có thể khá lớn
SEED = 2
# ===========================

assert RUN_SPLIT == 'test', 'Notebook này dành riêng FMB test, không sử dụng train'
assert MAX_IMAGES is None or (isinstance(MAX_IMAGES, int) and MAX_IMAGES > 0)
EXPECTED_COUNTS = {'train': 1220, 'test': 280}
REPO_URL = 'https://github.com/RollingPlain/SAGE_IVIF.git'
# Commit + checkpoint SHA256 lấy từ notebook Lộc bé đã chạy thành công.
PINNED_COMMIT = '377fdbef6b8b6ccbd1139358ad66bb41ea084dc1'
EXPECTED_CKPT_SHA256 = 'ecfde477ca4cd65401f207d5dc91ce6c475a49891d768dbdb7a66366255ee836'
REPO_DIR = Path('/kaggle/working/SAGE_IVIF')
RUN_TIME_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ID = f'sage_fmb_{RUN_SPLIT}_student_{RUN_TIME_UTC}'
OUTPUT_DIR = Path('/kaggle/working') / RUN_ID
FUSED_DIR = OUTPUT_DIR / 'fused'
LOG_DIR = OUTPUT_DIR / 'logs'
FUSED_DIR.mkdir(parents=True, exist_ok=False)
LOG_DIR.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Hãy bật GPU: Kaggle → Settings → Accelerator → GPU.'
DEVICE = torch.device('cuda:0')
torch.cuda.manual_seed_all(SEED)
print('SPLIT:', RUN_SPLIT, '| MAX_IMAGES:', MAX_IMAGES, '| RUN_ID:', RUN_ID)
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
print('Output:', OUTPUT_DIR)

## 2. Kiểm tra đúng thư mục FMB do bạn đã Add Input

Chấp nhận tên `Visible/Infrared` **hoặc** `Vis/Ir`. Không lấy ảnh từ Hugging Face, không tự resize, không thay đổi dữ liệu nguồn. Nếu mount Kaggle có đường dẫn ngắn hơn thì chỉ thử alias tương đương trong cùng dataset.

In [ ]:
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
IR_ALIASES = {'ir', 'infrared', 'thermal'}
VIS_ALIASES = {'vis', 'visible', 'vi', 'rgb'}

def find_modality_dirs(root):
    root = Path(root)
    if not root.is_dir():
        return None
    children = [x for x in root.iterdir() if x.is_dir()]
    ir = [x for x in children if x.name.lower() in IR_ALIASES]
    vis = [x for x in children if x.name.lower() in VIS_ALIASES]
    if len(ir) != 1 or len(vis) != 1:
        return None
    return ir[0], vis[0]

def resolve_fmb_root(test_root, split):
    assert split == 'test'
    # Ưu tiên đúng đường dẫn user xác nhận; chỉ thử Kaggle alias cùng dataset.
    # Không tìm train và không tự dùng dataset khác.
    paths = [test_root, Path('/kaggle/input/data-fmbtest/test')]
    for candidate in dict.fromkeys(paths):
        if find_modality_dirs(candidate):
            print('Dùng dataset:', candidate)
            return candidate
    details = '\n'.join(f'- {p} (exists={p.exists()})' for p in paths)
    raise FileNotFoundError(
        f'Không tìm thấy 2 thư mục IR/VIS ở split {split}. Đã thử:\n{details}\n'
        'Hãy Add Input FMB vào Kaggle và kiểm tra lại đường dẫn trong FMB_TEST_ROOT. '
        'Không dùng tập train thay thế test.'
    )

DATA_ROOT = resolve_fmb_root(FMB_TEST_ROOT, RUN_SPLIT)
IR_DIR, VIS_DIR = find_modality_dirs(DATA_ROOT)
print('IR_DIR =', IR_DIR, '| VIS_DIR =', VIS_DIR)
print('Các thư mục khác:', sorted(x.name for x in DATA_ROOT.iterdir() if x.is_dir()))

## 3. Audit **tất cả** cặp ảnh trong split, tạo manifest

Bắt buộc khớp stem giữa VIS và IR; kiểm tra từng file có thể đọc và kích thước đúng nhau. Kiểm tra trùng stem, lỗi ghép cặp và số ảnh. Với FMB gốc, kỳ vọng 1.220 train / 280 test. Cell này dừng nếu dữ liệu sai; không âm thầm bỏ ảnh lỗi.

In [ ]:
def map_by_stem(directory):
    files = sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS)
    mapping, duplicates = {}, []
    for p in files:
        if p.stem in mapping:
            duplicates.append(p.stem)
        else:
            mapping[p.stem] = p
    if duplicates:
        raise ValueError(f'Trùng stem trong {directory}: {duplicates[:20]}')
    return mapping

ir_map, vis_map = map_by_stem(IR_DIR), map_by_stem(VIS_DIR)
ir_only = sorted(set(ir_map) - set(vis_map))
vis_only = sorted(set(vis_map) - set(ir_map))
assert not ir_only and not vis_only, f'Lỗi ghép cặp: chỉ IR={ir_only[:10]}, chỉ VIS={vis_only[:10]}'
all_stems = sorted(set(ir_map) & set(vis_map))
expected = EXPECTED_COUNTS[RUN_SPLIT]
print('Số cặp khớp filename:', len(all_stems), '| FMB gốc kỳ vọng:', expected)
if STRICT_COUNT:
    assert len(all_stems) == expected, (
        f'{RUN_SPLIT}: tìm thấy {len(all_stems)} cặp, kỳ vọng {expected}. '
        'Hãy kiểm tra dataset/version/split, hoặc tắt STRICT_COUNT nếu cố ý dùng một phiên bản khác.'
    )

rows, invalid = [], []
for stem in tqdm(all_stems, desc='Kiểm tra input FMB'):
    ir_path, vis_path = ir_map[stem], vis_map[stem]
    ir_im = cv2.imread(str(ir_path), cv2.IMREAD_COLOR)
    vis_im = cv2.imread(str(vis_path), cv2.IMREAD_COLOR)
    if ir_im is None or vis_im is None or ir_im.shape[:2] != vis_im.shape[:2]:
        invalid.append({'stem':stem, 'ir_shape':None if ir_im is None else ir_im.shape,
                        'vis_shape':None if vis_im is None else vis_im.shape})
        continue
    h, w = vis_im.shape[:2]
    rows.append({'stem':stem, 'split':RUN_SPLIT, 'ir_path':str(ir_path),
                 'vis_path':str(vis_path), 'width':int(w), 'height':int(h),
                 'ir_extension':ir_path.suffix.lower(), 'vis_extension':vis_path.suffix.lower()})
assert not invalid, f'{len(invalid)} ảnh lỗi hoặc khác kích thước: {invalid[:8]}'
assert len(rows) == len(all_stems), 'Bị mất ảnh trong lúc audit'
all_manifest = pd.DataFrame(rows)
all_manifest.to_csv(OUTPUT_DIR / 'manifest_all.csv', index=False)
selected = rows if MAX_IMAGES is None else rows[:MAX_IMAGES]
assert selected, 'Không có ảnh hợp lệ'
selected_df = pd.DataFrame(selected)
selected_df.to_csv(OUTPUT_DIR / 'manifest.csv', index=False)
FULL_RUN = (MAX_IMAGES is None and len(selected) == len(all_stems))
print(f'INPUT: {len(all_stems)} cặp hợp lệ | ĐƯỢC CHỌN: {len(selected)} cặp | FULL_RUN={FULL_RUN}')
print('Các kích thước:', all_manifest.groupby(['width','height']).size().to_dict())
display(selected_df.head(5))

## 4. Lấy mã nguồn chính thức SAGE và khóa đúng checkpoint đã được Lộc bé kiểm thử

Chạy `git clone` lần đầu (cần Kaggle Internet). Nếu repo có sẵn, tái sử dụng. Checkout **đúng commit** của notebook Lộc bé và kiểm tra SHA-256 checkpoint. Không dùng `strict=False`. Nếu source/checkpoint khác, notebook **dừng** thay vì tuyên bố tái lập cùng baseline.

In [ ]:
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
current = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
if current != PINNED_COMMIT:
    print('Đang checkout commit chuẩn của notebook Lộc bé:', PINNED_COMMIT)
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','--depth','1','origin',PINNED_COMMIT],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',PINNED_COMMIT],check=True)
source_commit = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
assert source_commit == PINNED_COMMIT
checkpoint = REPO_DIR / 'checkpoints' / 'checkpoints.pt'
assert checkpoint.is_file(), f'Không thấy pretrained Student: {checkpoint}'
with checkpoint.open('rb') as f:
    ckpt_sha256 = hashlib.file_digest(f, 'sha256').hexdigest()
assert ckpt_sha256 == EXPECTED_CKPT_SHA256, (
    f'Checkpoint khác notebook smoke! got={ckpt_sha256}, expected={EXPECTED_CKPT_SHA256}. '
    'Không sử dụng checkpoint khác mà không ghi rõ provenance.'
)
assert (REPO_DIR/'dataset'/'dataset_test.py').is_file()
assert (REPO_DIR/'model_sub'/'model.py').is_file()
print('Source commit:', source_commit)
print('Checkpoint SHA256:', ckpt_sha256)
print('Checkpoint size MiB:', round(checkpoint.stat().st_size / 1024**2,3))

## 5. Adapter tên thư mục (không sao chép hoặc sửa ảnh FMB)

Dataset loader của SAGE yêu cầu `Ir/` và `Vis/`. Nếu dữ liệu của Mân có `Infrared/` và `Visible/`, tạo **symlink** trong `/kaggle/working`; thư mục gốc `/kaggle/input/...` vẫn giữ nguyên, chỉ đọc.

In [ ]:
STAGE_DIR = Path('/kaggle/working') / f'_sage_fmb_input_adapter_{RUN_SPLIT}'
STAGE_DIR.mkdir(parents=True, exist_ok=True)
for alias, target in [('Ir',IR_DIR),('Vis',VIS_DIR)]:
    link = STAGE_DIR / alias
    if link.is_symlink():
        assert link.resolve() == target.resolve(), f'Symlink cũ đang trỏ nơi khác: {link}'
    elif link.exists():
        raise RuntimeError(f'{link} tồn tại nhưng không phải symlink. Không tự xóa dữ liệu.')
    else:
        link.symlink_to(target.resolve(), target_is_directory=True)
    assert link.is_dir()
print('Adapter:',STAGE_DIR, '\n  Ir ->', (STAGE_DIR/'Ir').resolve(), '\n  Vis ->', (STAGE_DIR/'Vis').resolve())

## 6. Load pretrained Student và kiểm tra cấu hình model

**Giữ nguyên đường suy luận từ notebook Lộc bé:** `Data(mode='test')` (nghĩa là **không augment**, kể cả khi đang đọc split `train`), chuyển Visible sang OpenCV `YCrCb`, đưa `Y` và `IR` vào Student, sau đó ghép lại chroma Visible qua hàm `utils.YCrCb2RGB` của tác giả.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
from model_sub.model import Network
from dataset.dataset_test import Data
import utils

model = Network()
state = torch.load(checkpoint, map_location='cpu', weights_only=True)
assert isinstance(state, dict), f'Checkpoint format sai: {type(state)}'
model.load_state_dict(state, strict=True)
model.to(DEVICE).eval()
params = sum(p.numel() for p in model.parameters())
assert params == 135617, f'Student parameter count khác notebook Lộc bé: {params}'
torch.backends.cudnn.benchmark = True  # dùng cùng thiết lập với baseline smoke
print('Student pretrained: strict=True OK | params:', params)

# Dùng loader CHÍNH THỨC nhưng cố định thứ tự filenames (source gốc dùng set không ổn định).
dataset = Data(mode='test', img_dir=str(STAGE_DIR))
dataset.img_list = [r['stem'] for r in selected]
assert len(dataset) == len(selected)
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
print('Số ảnh cần inference:', len(dataset))

## 7. Chạy inference toàn bộ split và lưu fused RGB PNG

Đo `forward_ms` bằng đồng bộ CUDA; `fusion_ms` bao gồm forward + đổi YCrCb→RGB; `total_ms` còn gồm truyền GPU và lưu file (không gồm thời gian DataLoader chờ lấy batch). Warm-up không tính trong thống kê.

In [ ]:
torch.cuda.reset_peak_memory_stats(DEVICE)
with torch.inference_mode():
    sample = dataset[0]
    warm_y = sample['y'].unsqueeze(0).to(DEVICE)
    warm_ir = sample['ir'].unsqueeze(0).to(DEVICE)
    for _ in range(3):
        _ = model(warm_y, warm_ir)
    torch.cuda.synchronize(DEVICE)

records = []
loop_start = time.perf_counter()
with torch.inference_mode():
    for ix, batch in enumerate(tqdm(loader, total=len(dataset), desc=f'SAGE {RUN_SPLIT}')):
        item = selected[ix]
        stem = batch['name'][0]
        assert stem == item['stem'], f'Không khớp thứ tự image IDs tại {ix}: {stem} != {item["stem"]}'
        row_t0 = time.perf_counter()
        y = batch['y'].to(DEVICE, non_blocking=True)
        ir = batch['ir'].to(DEVICE, non_blocking=True)
        cb = batch['cb'].to(DEVICE, non_blocking=True)
        cr = batch['cr'].to(DEVICE, non_blocking=True)
        torch.cuda.synchronize(DEVICE)
        forward_t0 = time.perf_counter()
        fused_y, _ = model(y, ir)
        torch.cuda.synchronize(DEVICE)
        forward_t1 = time.perf_counter()
        fused_rgb = utils.YCrCb2RGB(torch.cat((fused_y,cb,cr),dim=1)).clamp(0,1)
        torch.cuda.synchronize(DEVICE)
        color_t1 = time.perf_counter()
        assert torch.isfinite(fused_rgb).all(), f'NaN/Inf output: {stem}'
        assert fused_rgb.ndim == 4 and fused_rgb.shape[1] == 3, f'Không phải ảnh RGB: {stem}'
        save_path = FUSED_DIR / f'{stem}.png'
        assert not save_path.exists(), f'Không ghi đè output: {save_path}'
        torchvision.utils.save_image(fused_rgb, str(save_path))
        end_t = time.perf_counter()
        assert save_path.is_file(), f'Lưu thất bại: {save_path}'
        records.append({
            'stem': stem, 'split': RUN_SPLIT, 'input_width': item['width'],
            'input_height': item['height'], 'fused_path':str(save_path),
            'forward_ms':round((forward_t1-forward_t0)*1000,3),
            'fusion_ms':round((color_t1-forward_t0)*1000,3),
            'total_ms':round((end_t-row_t0)*1000,3),
        })
loop_seconds = time.perf_counter()-loop_start
inference_df = pd.DataFrame(records)
inference_df.to_csv(OUTPUT_DIR / 'inference.csv', index=False)
assert len(inference_df)==len(selected), 'Thiếu kết quả inference'
print('Đã chạy:',len(inference_df),'ảnh | thời gian loop (s):',round(loop_seconds,2))
print('Forward mean/median/p95 (ms):', round(inference_df.forward_ms.mean(),3),
      round(inference_df.forward_ms.median(),3),
      round(float(np.percentile(inference_df.forward_ms,95)),3))
print('Peak GPU allocated MiB:',round(torch.cuda.max_memory_allocated(DEVICE)/1024**2,2))
display(inference_df.head(8))

## 8. Hậu kiểm **tất cả** ảnh fused và tạo bảng QA

Kiểm tra số lượng output, tên file, ảnh RGB PNG, kích thước và thống kê pixel. **Qua QA không đồng nghĩa chất lượng fusion tốt hơn** — vẫn phải tính metrics trên cùng dataset và protocol.

In [ ]:
outputs = sorted(FUSED_DIR.glob('*.png'))
actual_stems = {p.stem for p in outputs}
expected_stems = {r['stem'] for r in selected}
assert actual_stems == expected_stems, (
    f'Output thiếu: {sorted(expected_stems-actual_stems)[:10]}, '
    f'thừa: {sorted(actual_stems-expected_stems)[:10]}'
)
qa_rows, qa_failures = [], []
for item in tqdm(selected, desc='Kiểm tra ảnh output'):
    path = FUSED_DIR / f'{item["stem"]}.png'
    try:
        with Image.open(path) as im:
            assert im.format == 'PNG' and im.mode == 'RGB'
            assert im.size == (item['width'], item['height'])
            array = np.asarray(im)
            assert array.dtype == np.uint8 and array.ndim == 3 and array.shape[-1] == 3
            qa_rows.append({'stem':item['stem'], 'width':item['width'], 'height':item['height'],
                            'mode':im.mode, 'format':im.format,
                            'min_pixel':int(array.min()), 'max_pixel':int(array.max()),
                            'mean_pixel':round(float(array.mean()),4),
                            'std_pixel':round(float(array.std()),4),
                            'file_bytes':path.stat().st_size, 'status':'PASS'})
    except Exception as exc:
        qa_failures.append({'stem':item['stem'], 'error':str(exc)})
qa_df = pd.DataFrame(qa_rows)
qa_df.to_csv(OUTPUT_DIR / 'qa_per_image.csv', index=False)
(OUTPUT_DIR / 'qa_failures.json').write_text(json.dumps(qa_failures, indent=2, ensure_ascii=False),encoding='utf-8')
assert not qa_failures, f'QA FAIL: {qa_failures[:8]}'
assert len(qa_df)==len(selected)
qa_summary = {'split':RUN_SPLIT, 'input_pairs':len(all_stems),
              'selected_pairs':len(selected), 'fused_pngs':len(outputs),
              'qa_pass':len(qa_df), 'qa_fail':len(qa_failures),
              'is_full_split':FULL_RUN, 'expected_official_split_size':EXPECTED_COUNTS[RUN_SPLIT]}
(OUTPUT_DIR/'qa_summary.json').write_text(json.dumps(qa_summary,indent=2,ensure_ascii=False),encoding='utf-8')
print('QA PASS:', len(qa_df), '/', len(selected), 'ảnh RGB PNG đúng tên và kích thước')
display(qa_df.head(5))

## 9. Minh họa bốn bộ IR / Visible / Fused (không phải bằng chứng định lượng)

In [ ]:
show_items = selected[:min(4,len(selected))]
fig, axes = plt.subplots(len(show_items), 3, figsize=(14,3.4*len(show_items)), squeeze=False)
for j, item in enumerate(show_items):
    with Image.open(item['ir_path']) as im:
        ir_preview = im.convert('L').copy()
    with Image.open(item['vis_path']) as im:
        vis_preview = im.convert('RGB').copy()
    with Image.open(FUSED_DIR/f'{item["stem"]}.png') as im:
        fused_preview = im.convert('RGB').copy()
    for k, (im,title) in enumerate(zip([ir_preview,vis_preview,fused_preview], ['IR','Visible','SAGE Fused'])):
        axes[j,k].imshow(im, cmap='gray' if k==0 else None)
        axes[j,k].set_title(f'{item["stem"]} | {title}')
        axes[j,k].axis('off')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'preview_ir_vis_fused.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Ghi cấu hình, phiên bản và báo cáo phạm vi tái lập

In [ ]:
try:
    gpu_driver = subprocess.check_output([
        'nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'
    ], text=True, stderr=subprocess.STDOUT).strip()
except Exception as exc:
    gpu_driver = f'Unavailable: {exc}'
config = {
    'run_id':RUN_ID, 'source':'SAGE pretrained Student', 'split':RUN_SPLIT,
    'input_test_root_user_provided':str(FMB_TEST_ROOT),
    'actual_data_root':str(DATA_ROOT), 'actual_ir_dir':str(IR_DIR),
    'actual_vis_dir':str(VIS_DIR), 'expected_count':EXPECTED_COUNTS[RUN_SPLIT],
    'input_pairs':len(all_stems), 'max_images':MAX_IMAGES,
    'selected_pairs':len(selected), 'full_split':FULL_RUN,
    'seed':SEED, 'batch_size':1, 'data_mode':'test (no augmentation)',
    'preprocessing':'OpenCV BGR->YCrCb, torchvision ToTensor; IR BGR->GRAY; no resize',
    'postprocessing':'Official utils.YCrCb2RGB, clamp[0,1], torchvision.utils.save_image PNG',
    'repository_url':REPO_URL, 'repo_commit':source_commit,
    'checkpoint_file':str(checkpoint), 'checkpoint_sha256':ckpt_sha256,
    'student_parameters':params,
}
(OUTPUT_DIR/'run_config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
environment = {
    'utc':datetime.now(timezone.utc).isoformat(),
    'python':sys.version, 'platform':platform.platform(),
    'torch':torch.__version__, 'torchvision':torchvision.__version__,
    'cuda':torch.version.cuda, 'cudnn':torch.backends.cudnn.version(),
    'gpu':torch.cuda.get_device_name(0), 'nvidia_smi':gpu_driver,
    'opencv':cv2.__version__, 'numpy':np.__version__,
    'pandas':pd.__version__, 'pillow':metadata.version('Pillow'),
    'peak_cuda_allocated_mib':round(torch.cuda.max_memory_allocated(DEVICE)/1024**2,3),
    'loop_total_seconds':round(loop_seconds,3),
    'mean_forward_ms':round(float(inference_df.forward_ms.mean()),3),
    'median_forward_ms':round(float(inference_df.forward_ms.median()),3),
    'p95_forward_ms':round(float(np.percentile(inference_df.forward_ms,95)),3),
    'mean_fusion_ms':round(float(inference_df.fusion_ms.mean()),3),
}
(OUTPUT_DIR/'environment.json').write_text(json.dumps(environment,indent=2,ensure_ascii=False),encoding='utf-8')
with (OUTPUT_DIR/'pip_freeze.txt').open('w',encoding='utf-8') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)

scope = 'FULL SPLIT' if FULL_RUN else 'SUBSET/SMOKE'
report = f"""# Báo cáo SAGE Student baseline — Quân

- Run ID: `{RUN_ID}`
- Dataset: `{DATA_ROOT}` (split **{RUN_SPLIT}**, {scope})
- FMB pairs: {len(all_stems)}; đã chạy: {len(selected)}; ảnh fused hợp lệ: {len(outputs)}
- Source commit: `{source_commit}`
- Checkpoint SHA256: `{ckpt_sha256}`
- GPU: {environment['gpu']}
- Student parameters: {params:,}
- Forward mean / median / p95 (ms): {environment['mean_forward_ms']} / {environment['median_forward_ms']} / {environment['p95_forward_ms']}
- Peak CUDA allocated (MiB): {environment['peak_cuda_allocated_mib']}
- QA: {len(qa_df)}/{len(selected)} PASS

**Phạm vi kết luận:** Pretrained Student inference thành công trên {len(selected)} ảnh {RUN_SPLIT} FMB. Không suy ra đã train lại Teacher/Student. Không khẳng định tái lập 61.2% FMB mIoU, vì còn cần downstream segmentation evaluation. Chưa có bảng EN/SD/SCD/MS-SSIM nếu chưa chạy riêng evaluation pipeline.

**Files:** manifest.csv, manifest_all.csv, inference.csv, qa_per_image.csv, qa_summary.json, run_config.json, environment.json, pip_freeze.txt, preview_ir_vis_fused.png, fused/*.png.
"""
(OUTPUT_DIR/'README_results.md').write_text(report,encoding='utf-8')
(LOG_DIR/'run.log').write_text(
    f'RUN_ID={RUN_ID}\nSPLIT={RUN_SPLIT}\nDATA_ROOT={DATA_ROOT}\n'
    f'EXPECTED_COUNT={EXPECTED_COUNTS[RUN_SPLIT]}\nSELECTED={len(selected)}\n'
    f'FUSED={len(outputs)}\nQA_PASS={len(qa_df)}\nQA_FAIL={len(qa_failures)}\n'
    f'COMMIT={source_commit}\nCHECKPOINT_SHA256={ckpt_sha256}\n'
    f'MEAN_FORWARD_MS={environment["mean_forward_ms"]}\n',
    encoding='utf-8'
)
print(report)

## 11. Nén output để tải từ Kaggle (tùy chọn)

ZIP có thể lớn nếu chạy đủ 280 ảnh. Giữ output lớn trên Kaggle/Drive; **không commit ảnh hoặc checkpoint lên GitHub nhóm**. Chỉ đưa notebook, manifest, JSON, log/bảng nhỏ và link artifact vào `xulianh`.

In [ ]:
if EXPORT_ZIP:
    zip_base = Path('/kaggle/working') / RUN_ID
    zip_path = Path(shutil.make_archive(str(zip_base)+'_results', 'zip', root_dir=OUTPUT_DIR))
    print('ZIP đã tạo:', zip_path)
    print('Kích thước ZIP MiB:', round(zip_path.stat().st_size/1024**2,2))
else:
    print('EXPORT_ZIP=False, kết quả giữ tại:', OUTPUT_DIR)
print('HOÀN THÀNH PRETRAINED STUDENT INFERENCE / QA:',len(outputs),'ảnh',RUN_SPLIT)
print('ĐỂ TÁI LẬP METRIC FUSION: chuyển manifest + fused sang evaluation pipeline của Lộc lớn.')

## Checklist nghiệm thu / nộp GitHub

- [ ] Đường dẫn dataset đúng FMB, đúng split `test`; đủ **280 cặp test** với `STRICT_COUNT=True`.
- [ ] Source commit và checkpoint SHA256 khớp notebook 10 ảnh của Lộc bé.
- [ ] `strict=True` load Student, 135.617 tham số.
- [ ] `qa_summary.json` báo `qa_fail=0`, `selected_pairs=fused_pngs`.
- [ ] `manifest.csv`, `inference.csv`, `run_config.json`, `environment.json`, `logs/run.log` tồn tại; `preview_ir_vis_fused.png` nhìn hợp lý.
- [ ] ZIP tải được từ Kaggle Output; **không push fused PNG/checkpoint** vào repo.
- [ ] Copy notebook này vào `notebooks/quan/SAGE_Quan_FMB_Kaggle_Full_Baseline.ipynb` của repo `xulianh` và tạo PR.
- [ ] Chuyển manifest và ảnh fused cho Lộc lớn đánh giá **EN/SD/SCD/MS-SSIM**, kiểm chứng protocol trước khi kết luận chất lượng.
- [ ] Nếu đã chạy trên **test**, không sử dụng kết quả đó để lựa chọn/chỉnh sửa solution của nhóm.

**Giới hạn:** Notebook chưa thực thi trên GPU Kaggle trong môi trường tạo file hiện tại; các kiểm tra thực nghiệm chỉ được xác nhận sau khi bạn Run All trên Kaggle.